# Risk/UQ Paper Track: UQ Benchmark

## Objective
Benchmark calibration, selective risk, and shift robustness using persisted risk-model artifacts.


## Hypotheses Being Tested
- **H1:** Calibrated monitor improves ECE vs raw monitor on `nominal_clean`.
- **H2:** Calibration gains persist on most configured shift suites.
- **H3:** Selective risk curves improve with calibrated probabilities.


## Methodology
1. Deterministic bootstrap and run-context resolution.
2. Validate upstream contract gates/stages from training notebook.
3. Execute resume-aware UQ benchmark flow.
4. Export diagnostics and update manifest.


## Step 1 - Bootstrap Environment


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Required Config Block + Run Context


In [ ]:
from src.workflows import initialize_run_context, report_run_context

RUN_TAG = ''
RUN_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'

N_SHARDS = 1
SHARD_ID = 'auto'

AUTO_RUN_MAIN_LOOP_WHEN_READY = False
RUN_MAIN_LOOP_OVERRIDE = False

RUN_MODE = 'auto'  # auto | fresh | resume
WARN_ON_CONFIG_DRIFT = True

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
RUN_TAG = run_context.run_tag
SHARD_ID = run_context.shard_id
run_prefix = run_context.run_prefix

print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)

# Benchmark knobs
cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg.uq_eval_probability_bins = 15
cfg


## Step 3 - Contract Gate Check (Upstream Training)
This notebook requires `risk_training_completed` from the training notebook manifest.


In [ ]:
from src.workflows import (
    load_notebook_contract_manifest,
    validate_notebook_contract_manifest,
    write_notebook_contract_manifest,
)

manifest = load_notebook_contract_manifest(cfg.run_prefix)
ok, reasons = validate_notebook_contract_manifest(
    manifest,
    require_quick_probe=True,
    require_preflight=True,
    required_stages=('risk_training_completed',),
)
if not ok:
    raise RuntimeError(
        'Notebook contract prerequisites not met. Run risk_model_training_colab.ipynb first. '
        f'Reasons: {reasons}'
    )

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='uq_benchmark_colab',
    stage='uq_benchmark_started',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
)
print('manifest_path =', manifest_path)


## Step 4 - Execute Resume-Aware Benchmark Flow


In [ ]:
from src.workflows import load_existing_risk_dataset_artifact, run_uq_benchmark_flow

dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
if dataset_df.empty:
    raise RuntimeError('Missing persisted risk dataset artifact. Run training notebook first.')

uq_bundle = run_uq_benchmark_flow(
    cfg=cfg,
    dataset_df=dataset_df,
    run_prefix=cfg.run_prefix,
    resume_mode=RUN_MODE,
)

print('loaded_from_existing =', uq_bundle.loaded_from_existing)
display(uq_bundle.benchmark_bundle.summary_df)


## Step 5 - Evaluation and Diagnostics


In [ ]:
display(uq_bundle.benchmark_bundle.per_shift_df.head(30))
display(uq_bundle.benchmark_bundle.reliability_df.head(30))
display(uq_bundle.benchmark_bundle.selective_curve_df.head(30))


## Step 6 - Update Manifest (Benchmark Completed)


In [ ]:
manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='uq_benchmark_colab',
    stage='uq_benchmark_completed',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields={
        'loaded_from_existing': int(bool(uq_bundle.loaded_from_existing)),
        'summary_rows': int(len(uq_bundle.benchmark_bundle.summary_df)),
        'per_shift_rows': int(len(uq_bundle.benchmark_bundle.per_shift_df)),
    },
)
print('manifest_path =', manifest_path)
